# Market Basics

Markets connect company fundamentals with investor expectations. This notebook introduces the basic market data concepts needed before working with open-high-low-close-volume data, returns, indicators, and portfolio construction.

Abbreviations used in this notebook:

- **OHLCV**: Open, High, Low, Close, Volume; the standard daily market data fields.
- **VWAP**: Volume-Weighted Average Price, the average traded price weighted by volume.
- **CHF**: Swiss franc, the currency used in the examples.

## 1. Intuition

A stock price is the market's current clearing price for ownership in a company. It changes when buyers and sellers update their expectations about future cash flows, risk, interest rates, liquidity, or sentiment.

Key market concepts:

- **Price**: the last traded value of one share.
- **Market capitalization**: total equity value implied by the share price.
- **Bid and ask**: the best price buyers offer and sellers request.
- **Spread**: the transaction cost between bid and ask.
- **Volume**: number of shares traded.
- **Return**: percentage change in value over a period.
- **Volatility**: variation in returns, often used as a risk proxy.

Fundamentals help estimate value. Markets show how investors price that value right now.

## 2. Mathematics

Market capitalization:

$$
\text{Market Cap} = \text{Share Price} \times \text{Shares Outstanding}
$$

Where:

- $\text{Shares Outstanding}$ = number of shares issued and outstanding
- $\text{Share Price}$ = market price per share
- $\text{Market Cap}$ = equity market value

Simple return:

$$
R_t = \frac{P_t - P_{t-1}}{P_{t-1}}
$$

Where:

- $R_t$ = simple return at time $t$
- $P_t$ = price at time $t$
- $P_{t-1}$ = price in the previous period
- $t$ = time period index

Log return:

$$
r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)
$$

Where:

- $r_t$ = log return at time $t$
- $P_t$ = price at time $t$
- $P_{t-1}$ = price in the previous period
- $t$ = time period index

Bid-ask spread:

$$
\text{Spread} = Ask - Bid
$$

Where:

- $Ask$ = lowest price sellers are willing to accept
- $Bid$ = highest price buyers are willing to pay
- $Spread$ = ask price minus bid price

Relative spread:

$$
\text{Relative Spread} = \frac{Ask - Bid}{Midpoint}
$$

Where:

- $Ask$ = lowest price sellers are willing to accept
- $Bid$ = highest price buyers are willing to pay
- $Midpoint$ = average of bid and ask prices
- $Spread$ = ask price minus bid price

Volume-weighted average price:

$$
VWAP = \frac{\sum_t Price_t \times Volume_t}{\sum_t Volume_t}
$$

Where:

- $VWAP$ = volume-weighted average price
- $Price_t$ = transaction or close price at time $t$
- $Volume_t$ = traded volume at time $t$
- $t$ = time period index

Annualized volatility from daily returns:

$$
\sigma_{annual} = \sigma_{daily} \times \sqrt{252}
$$

Where:

- $\sigma_{annual}$ = annualized volatility
- $\sigma_{daily}$ = daily volatility
- $252$ = approximate number of trading days in one year

## 3. Implementation

We will create a small synthetic market dataset for a defensive consumer company. The goal is not price prediction; the goal is to understand market data fields and the calculations built on top of them.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
pd.options.display.float_format = "{:,.4f}".format
np.random.seed(7)

trading_days = pd.bdate_range("2024-01-02", periods=90)
start_price = 105.0
shares_outstanding = 2_650_000_000

returns = np.random.normal(loc=0.00035, scale=0.011, size=len(trading_days))
close = start_price * np.cumprod(1 + returns)
open_price = close * (1 + np.random.normal(0, 0.003, len(trading_days)))
high = np.maximum(open_price, close) * (1 + np.random.uniform(0.001, 0.012, len(trading_days)))
low = np.minimum(open_price, close) * (1 - np.random.uniform(0.001, 0.012, len(trading_days)))
volume = np.random.randint(1_200_000, 4_800_000, len(trading_days))
spread = np.random.uniform(0.02, 0.12, len(trading_days))

market = pd.DataFrame({
    "date": trading_days,
    "open": open_price,
    "high": high,
    "low": low,
    "close": close,
    "volume": volume,
    "bid": close - spread / 2,
    "ask": close + spread / 2,
})

market["midpoint"] = (market["bid"] + market["ask"]) / 2
market["simple_return"] = market["close"].pct_change()
market["log_return"] = np.log(market["close"] / market["close"].shift(1))
market["market_cap"] = market["close"] * shares_outstanding
market["spread"] = market["ask"] - market["bid"]
market["relative_spread"] = market["spread"] / market["midpoint"]
market["traded_value"] = market["close"] * market["volume"]

market.head().style.format({
    "open": "CHF {:,.2f}",
    "high": "CHF {:,.2f}",
    "low": "CHF {:,.2f}",
    "close": "CHF {:,.2f}",
    "volume": "{:,.0f}",
    "bid": "CHF {:,.2f}",
    "ask": "CHF {:,.2f}",
    "midpoint": "CHF {:,.2f}",
    "simple_return": "{:.2%}",
    "log_return": "{:.2%}",
    "market_cap": "CHF {:,.0f}",
    "spread": "CHF {:,.2f}",
    "relative_spread": "{:.3%}",
    "traded_value": "CHF {:,.0f}",
})

The most common daily market data format is **OHLCV**, which means open, high, low, close, and volume. These fields are the raw material for later notebooks on indicators and backtesting.

In [ ]:
summary_display = pd.DataFrame({
    "metric": [
        "Start close",
        "End close",
        "Total return",
        "Average daily return",
        "Daily volatility",
        "Annualized volatility",
        "Average volume",
        "Average relative spread",
        "Latest market cap",
    ],
    "value": [
        f"CHF {market['close'].iloc[0]:,.2f}",
        f"CHF {market['close'].iloc[-1]:,.2f}",
        f"{market['close'].iloc[-1] / market['close'].iloc[0] - 1:.1%}",
        f"{market['simple_return'].mean():.3%}",
        f"{market['simple_return'].std():.2%}",
        f"{market['simple_return'].std() * np.sqrt(252):.1%}",
        f"{market['volume'].mean():,.0f} shares",
        f"{market['relative_spread'].mean():.3%}",
        f"CHF {market['market_cap'].iloc[-1] / 1_000_000_000:,.1f}bn",
    ],
})
summary_display

In [ ]:
vwap = market["traded_value"].sum() / market["volume"].sum()
latest_close = market["close"].iloc[-1]

print(f"VWAP over sample: CHF {vwap:,.2f}")
print(f"Latest close: CHF {latest_close:,.2f}")
print(f"Latest market cap: CHF {market['market_cap'].iloc[-1] / 1_000_000_000:,.1f}bn")

## 4. Visualization

Market charts help reveal price trend, trading activity, and day-to-day return behavior. They are descriptive, not proof of intrinsic value.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]})

axes[0].plot(market["date"], market["close"], color="#2f6f8f", label="Close")
axes[0].fill_between(market["date"], market["low"], market["high"], color="#2f6f8f", alpha=0.18, label="Daily range")
axes[0].set_title("Synthetic Stock Price: Close and Daily Range")
axes[0].set_ylabel("CHF per share")
axes[0].legend(loc="upper left")

axes[1].bar(market["date"], market["volume"], color="#9a6b2f", alpha=0.8)
axes[1].set_title("Trading Volume")
axes[1].set_ylabel("Shares")
axes[1].set_xlabel("Date")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(market["simple_return"].dropna(), bins=18, color="#2f6f8f", edgecolor="white")
axes[0].axvline(0, color="black", linewidth=1)
axes[0].set_title("Daily Return Distribution")
axes[0].set_xlabel("Daily return")
axes[0].set_ylabel("Count")
axes[0].xaxis.set_major_formatter(lambda x, pos: f"{x:.1%}")

axes[1].plot(market["date"], market["relative_spread"], color="#9a6b2f")
axes[1].set_title("Relative Bid-Ask Spread")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Spread / Midpoint")
axes[1].yaxis.set_major_formatter(lambda x, pos: f"{x:.2%}")

plt.tight_layout()
plt.show()

## 5. Application

When looking at a stock such as Nestle, market basics help connect company analysis to tradable reality:

- Market cap tells us the equity value implied by the current share price.
- Returns show how the price changed over time.
- Volatility gives a first approximation of market risk.
- Volume helps us understand liquidity.
- Bid-ask spread indicates transaction cost and trading friction.
- VWAP is useful when evaluating execution quality over a trading window.

This is the bridge from fundamental analysis to market data. In later notebooks, these fields become inputs for indicators, portfolio risk, and backtests.

In [ ]:
latest = market.iloc[-1]
prior = market.iloc[0]

interpretation = pd.DataFrame({
    "metric": [
        "Price change",
        "Total return",
        "Latest bid",
        "Latest ask",
        "Latest spread",
        "Latest relative spread",
        "Latest traded value",
    ],
    "value": [
        f"CHF {latest['close'] - prior['close']:,.2f}",
        f"{latest['close'] / prior['close'] - 1:.1%}",
        f"CHF {latest['bid']:,.2f}",
        f"CHF {latest['ask']:,.2f}",
        f"CHF {latest['spread']:,.2f}",
        f"{latest['relative_spread']:.3%}",
        f"CHF {latest['traded_value'] / 1_000_000:,.1f}m",
    ],
})

interpretation

## 6. Reflection

- Price is what the market currently pays; value is what analysis tries to estimate.
- Market cap converts a per-share price into total equity value.
- Returns, not price levels, are the usual input for risk and portfolio analysis.
- Volume and spread matter because investing also depends on liquidity and transaction costs.
- OHLCV data is the foundation for market data pipelines, technical indicators, and backtests.

Questions to answer after running the notebook:

1. Why is market cap more useful than share price alone?
2. Why do analysts usually study returns instead of raw prices?
3. What does a wider bid-ask spread tell you about liquidity?
4. How can a stock price move even when company fundamentals have not changed?